In [ ]:
import pandas as pd
import torch
import numpy as np
from datasets import Dataset
from transformers import T5Tokenizer, Seq2SeqTrainingArguments, Seq2SeqTrainer, T5ForConditionalGeneration
import evaluate
df = pd.read_json('/kaggle/input/datasets/caokhoihuynh/xl-sum/XLSum_complete_v2.0/english_train.jsonl', lines=True)
print(df.columns)
print(df.shape)
print(df.iloc[0]['text'][:300])
print('---')
print(df.iloc[0]['summary'])

df = df.head(50000)
print(df.shape)
print(f'Средняя длина text: {df['text'].str.len().mean().round(0)}')
print(f'Средняя длина summary: {df['summary'].str.len().mean().round(0)}')

tokenizer = T5Tokenizer.from_pretrained('t5-small')


def preprocess(ex):
    inputs = tokenizer(
        ['summarize: ' + t for t in ex['text']],
        max_length=718,
        truncation=True,
        padding='max_length'
    )

    targets = tokenizer(
        ex['summary'],
        max_length=128,
        truncation=True,
        padding='max_length'
    )
    inputs['labels'] = targets['input_ids']
    return inputs


dataset = Dataset.from_pandas(df[['text', 'summary']])
dataset = dataset.map(preprocess, batched=True, remove_columns=['text', 'summary'])

print(dataset)
print(dataset[0].keys())

dataset = dataset.train_test_split(test_size = 0.1, seed=42)
train_dataset = dataset['train']
val_dataset = dataset['test']
print(f'train: {len(train_dataset)}, val: {len(val_dataset)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = T5ForConditionalGeneration.from_pretrained('t5-small')
print(f'Параметров: {model.num_parameters()}')

rouge = evaluate.load('rouge')


def compute_metrics(eval_pred):
    predict, labels = eval_pred
    predict = np.where(predict != -100, predict, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    predict = tokenizer.batch_decode(predict, skip_special_tokens=True)
    labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=predict, references=labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}


train_args = Seq2SeqTrainingArguments(
    output_dir='./results',
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    warmup_steps=500,
    eval_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    fp16=True,
    logging_steps=100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

test_df = pd.read_json('/kaggle/input/datasets/caokhoihuynh/xl-sum/XLSum_complete_v2.0/english_test.jsonl', lines=True)
test_df = test_df.head(1000)

test_dataset = Dataset.from_pandas(test_df[['text', 'summary']])
test_dataset = test_dataset.map(preprocess, batched=True, remove_columns=['text', 'summary'])

predictions = trainer.predict(test_dataset)
metrics = compute_metrics((predictions.predictions, predictions.label_ids))
print(metrics)

trainer.save_model('t5_trained')
tokenizer.save_pretrained('t5_trained')

preds = np.where(predictions.predictions != -100, predictions.predictions, tokenizer.pad_token_id)
labels = np.where(predictions.label_ids != -100, predictions.label_ids, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

scored = []
for i in range(len(decoded_preds)):
    result = rouge.compute(predictions=[decoded_preds[i]], references=[decoded_labels[i]], use_stemmer=True)
    scored.append({
        'idx': i,
        'score': result['rougeL'],
        'pred': decoded_preds[i],
        'ref': decoded_labels[i],
        'source': tokenizer.decode(test_dataset[i]['input_ids'], skip_special_tokens=True)
    })

scored.sort(key=lambda x: x['score'])


def print_example(example, title):
    print('=' * 70)
    print(title, f'(ROUGE-L: {example["score"]:.4f})')
    print('=' * 70)
    print(f'Статья:\n{example["source"][:350]}...')
    print('-' * 70)
    print(f'Reference:\n{example["ref"]}')
    print('-' * 70)
    print(f'Generated:\n{example["pred"]}')
    print('=' * 70)
    print()


print('\nХудшие примеры\n')
for ex in scored[:2]:
    print_example(ex, 'ПРОВАЛ')

print('\nЛучшие примеры\n')
for ex in scored[-2:]:
    print_example(ex, 'УСПЕХ')

